# 3D CUDA-Accelerated Reverse Time Migration (RTM)

This notebook compiles and runs the repository's native CUDA C++ implementation, then visualizes the synthetic RTM image and receiver traces. It is an educational prototype using a small 64 by 64 by 64 grid, not a production seismic-processing package.

Before running, choose a GPU runtime in Colab from Runtime, Change runtime type, then select a GPU.

## 1. Check the environment

The CUDA C++ program requires a working NVIDIA GPU runtime and the NVIDIA CUDA compiler, nvcc. If either is missing, switch to a supported GPU runtime before continuing.

In [ ]:
import os
import subprocess
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)
compiler_check = subprocess.run(["nvcc", "--version"], check=False)
if compiler_check.returncode != 0:
    raise RuntimeError("nvcc is unavailable. Select a Colab GPU runtime with the CUDA Toolkit installed.")

## 2. Download the source

This cell clones the public repository into the Colab session. Re-run it safely if the directory already exists.

In [ ]:
repo_dir = Path("/content/3D-CUDA-RTM-Seismic-Imaging")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "https://github.com/Venkata-Sushanth/3D-CUDA-RTM-Seismic-Imaging.git", str(repo_dir)], check=True)
os.chdir(repo_dir)
print("Working directory:", Path.cwd())

## 3. Compile and run CUDA RTM

The GPU program models forward wave propagation, records synthetic receiver traces, then back-propagates the traces and accumulates a basic forward/backward wavefield correlation image. It reports timing for this run; do not assume a GPU is always faster for every grid size.

In [ ]:
subprocess.run(["nvcc", "-O3", "-std=c++17", "src/cuda_rtm.cu", "-o", "cuda_rtm"], check=True)
subprocess.run(["./cuda_rtm"], check=True)

## 4. Visualize generated outputs

The program writes CSV files under the results directory. The velocity and image plots show a middle-y slice of the 3D model. Values are synthetic and have arbitrary amplitude for this demonstration.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

velocity_df = pd.read_csv("results/cuda_velocity_model.csv")
image_df = pd.read_csv("results/cuda_rtm_image.csv")
traces_df = pd.read_csv("results/cuda_receiver_gather.csv")
NX = NY = NZ = 64
velocity = velocity_df["value"].to_numpy().reshape(NZ, NY, NX)
image = image_df["value"].to_numpy().reshape(NZ, NY, NX)
traces = traces_df["amplitude"].to_numpy().reshape(-1, 56)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
im0 = axes[0].imshow(velocity[:, NY // 2, :], origin="upper", aspect="auto")
axes[0].set_title("Synthetic velocity model")
axes[0].set_xlabel("x grid point")
axes[0].set_ylabel("Depth z grid point")
fig.colorbar(im0, ax=axes[0], shrink=0.8, label="Velocity (m/s)")
image_slice = image[:, NY // 2, :]
limit = np.percentile(np.abs(image_slice), 99.5) + 1e-12
im1 = axes[1].imshow(image_slice, origin="upper", aspect="auto", cmap="seismic", vmin=-limit, vmax=limit)
axes[1].set_title("RTM image (middle y slice)")
axes[1].set_xlabel("x grid point")
axes[1].set_ylabel("Depth z grid point")
fig.colorbar(im1, ax=axes[1], shrink=0.8, label="Correlation amplitude")
axes[2].imshow(traces.T, aspect="auto", origin="lower", cmap="seismic")
axes[2].set_title("Synthetic receiver gather")
axes[2].set_xlabel("Time step")
axes[2].set_ylabel("Receiver index")
plt.tight_layout()
plt.show()

## Limitations and report guidance

This is a simplified acoustic model with fixed-zero outer boundaries, synthetic receiver data, a second-order finite-difference stencil, and a basic correlation imaging condition. It is intended to demonstrate CUDA parallelization and the RTM workflow, not to prove the location of real faults or hydrocarbon reservoirs. For a fair CPU/GPU comparison, use the same grid and time-step count, record the hardware and compiler versions, compare numerical outputs, and repeat timing runs. GPU timing excludes neither all runtime setup effects nor every possible source of variability.